In [1]:
# Sel ini menghasilkan TIGA dataset cabang (kota) terpisah untuk Tugas Mandiri Pertemuan 3
import numpy as np
import pandas as pd

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)  # seed berbeda tiap kota agar datanya bervariasi, namun tetap konsisten/reproducible
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat: {df_cabang.shape[0]} baris")

print("\nKetiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.")

Berkas 'transaksi_magelang.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_semarang.csv' berhasil dibuat: 200 baris

Ketiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.


In [2]:
# Membuat struktur direktori HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/ecommerce/raw
!hdfs dfs -mkdir -p /user/mahasiswa/ecommerce/processed

# Menampilkan struktur direktori ecommerce 
!hdfs dfs -ls /user/mahasiswa/ecommerce

Found 2 items
drwxr-xr-x   - fais supergroup          0 2026-09-10 04:06 /user/mahasiswa/ecommerce/processed
drwxr-xr-x   - fais supergroup          0 2026-09-10 03:27 /user/mahasiswa/ecommerce/raw


In [3]:
# Menggunggah berkas CSV ke folder raw di HDFS
!hdfs dfs -put transaksi_magelang.csv /user/mahasiswa/ecommerce/raw/
!hdfs dfs -put transaksi_yogyakarta.csv /user/mahasiswa/ecommerce/raw/
!hdfs dfs -put transaksi_semarang.csv /user/mahasiswa/ecommerce/raw/

# Menampilkan isi direktori raw lengkap dengan informasi ukuran format yang mudah dibaca (-h)
!hdfs dfs -ls -h /user/mahasiswa/ecommerce/raw/

put: `/user/mahasiswa/ecommerce/raw/transaksi_magelang.csv': File exists
put: `/user/mahasiswa/ecommerce/raw/transaksi_yogyakarta.csv': File exists
put: `/user/mahasiswa/ecommerce/raw/transaksi_semarang.csv': File exists
Found 3 items
-rw-r--r--   1 fais supergroup     12.0 K 2026-09-10 03:27 /user/mahasiswa/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 fais supergroup     11.9 K 2026-09-10 03:27 /user/mahasiswa/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 fais supergroup     12.4 K 2026-09-10 03:27 /user/mahasiswa/ecommerce/raw/transaksi_yogyakarta.csv


In [4]:
import os
import subprocess
import pandas as pd

os.environ['CLASSPATH'] = subprocess.check_output(['hadoop', 'classpath', '--glob']).decode('utf-8').strip()

# Membaca data langsung dari HDFS menggunakan url HDFS (port default 9000)
url_magelang = 'hdfs://localhost:9000/user/mahasiswa/ecommerce/raw/transaksi_magelang.csv'
url_yogyakarta = 'hdfs://localhost:9000/user/mahasiswa/ecommerce/raw/transaksi_yogyakarta.csv'
url_semarang = 'hdfs://localhost:9000/user/mahasiswa/ecommerce/raw/transaksi_semarang.csv'

df_magelang = pd.read_csv(url_magelang)
df_yogyakarta = pd.read_csv(url_yogyakarta)
df_semarang = pd.read_csv(url_semarang)

# Menggabungkan ketiga dataframe menjadi satu
df_gabungan = pd.concat([df_magelang, df_yogyakarta, df_semarang], ignore_index=True)

# Membuktikan hasil gabungan dengan menghitung jumlah baris per kota
print("Jumlah transaksi per kota:")
print(df_gabungan['kota'].value_counts())

2026-09-10 09:16:11,615 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Jumlah transaksi per kota:
kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64


In [5]:
# 1. Tambahkan kolom `total_pendapatan` (`unit_terjual x harga_satuan`)
df_gabungan['total_pendapatan'] = df_gabungan['unit_terjual'] * df_gabungan['harga_satuan']
print("<========== 5 Baris Pertama Data Gabungan + Total Pendapatan ==========>\n")
print(df_gabungan.head())

# 2. Buatlah satu tabel ringkasan (`groupby`) yang menampilkan total pendapatan per kota dan per kategori sekaligus.
ringkasan = df_gabungan.groupby(['kota', 'kategori'])['total_pendapatan'].sum().reset_index()
print("\n<========== Ringkasan ==========>\n")
print(ringkasan)

df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
ringkasan.to_csv("ringkasan_kota_kategori.csv", index=False)

# 3. Simpan dua hasil berikut sebagai berkas CSV baru di disk lokal, lalu unggah keduanya ke `/user/[username]/ecommerce/processed` di HDFS:
# Menyimpan hasil olahan data ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/ecommerce/processed

!hdfs dfs -put -f data_gabungan_bersih.csv /user/mahasiswa/ecommerce/processed
!hdfs dfs -put -f ringkasan_kota_kategori.csv /user/mahasiswa/ecommerce/processed

!hdfs dfs -ls -h /user/mahasiswa/ecommerce/processed

<========== 5 Baris Pertama Data Gabungan + Total Pendapatan ==========>

   order_id     tanggal      kategori  unit_terjual  harga_satuan  \
0  MAG-2000  2026-08-12       Fashion             1         50000   
1  MAG-2001  2026-08-18    Elektronik             7         25000   
2  MAG-2002  2026-08-07    Elektronik             7         25000   
3  MAG-2003  2026-08-24  Rumah Tangga             6         25000   
4  MAG-2004  2026-08-30       Fashion             7        100000   

  metode_pembayaran      kota  total_pendapatan  
0     Transfer Bank  Magelang             50000  
1               COD  Magelang            175000  
2          E-Wallet  Magelang            175000  
3               COD  Magelang            150000  
4     Transfer Bank  Magelang            700000  

<========== Ringkasan ==========>

          kota                kategori  total_pendapatan
0     Magelang              Elektronik          18775000
1     Magelang                 Fashion          27750000
2   